# Representation Similarity Analysis

Compare representations across layers and positions: CKA, cosine similarity,
representation drift, effective dimensionality, and geometry summary.

## Why This Matters

Representation similarity characterizes the structure of internal model representations. Understanding how models organize information — which concepts are linearly separable, how representations cluster, and how they change across layers — is central to interpretability.

**Key references:**
- [Belinkov (2022) "Probing Classifiers"](https://arxiv.org/abs/2102.12452) — Methodology for probing neural representations
- [Kornblith et al. (2019) "Similarity of Neural Network Representations"](https://arxiv.org/abs/1905.00414) — CKA for comparing representations across models and layers

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.representation_similarity_analysis import (
    layer_representation_similarity, position_representation_similarity,
    representation_drift, representation_effective_dimension,
    representation_geometry_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Layer Representation Similarity (Cosine)

How similar are layer representations to each other?

In [ ]:
result = layer_representation_similarity(model, tokens, metric='cosine')
print(f"Adjacent similarity: {result['mean_adjacent_similarity']:.4f}")
print(f"Distant similarity: {result['mean_distant_similarity']:.4f}\n")
for i in range(result['n_layers']):
    row = ' '.join(f'{result["similarity_matrix"][i,j]:.3f}' for j in range(result['n_layers']))
    print(f'  Layer {i}: [{row}]')

## Layer Representation Similarity (CKA)

Centered Kernel Alignment captures structural similarity beyond simple cosine.

In [ ]:
result = layer_representation_similarity(model, tokens, metric='cka')
for i in range(result['n_layers']):
    row = ' '.join(f'{result["similarity_matrix"][i,j]:.3f}' for j in range(result['n_layers']))
    print(f'  Layer {i}: [{row}]')

## Position Representation Similarity

Are different token positions similar or diverse at a given layer?

In [ ]:
for layer in range(4):
    result = position_representation_similarity(model, tokens, layer=layer)
    flag = ' [DIVERSE]' if result['is_position_diverse'] else ''
    print(f"  Layer {layer}: mean_cos={result['mean_pairwise_similarity']:.4f}, "
          f"adj={result['mean_adjacent_similarity']:.4f}{flag}")

## Representation Drift

How does a single position's representation evolve across layers?

In [ ]:
result = representation_drift(model, tokens, position=-1)
print(f"Total drift: {result['total_drift']:.4f}")
print(f"Gradual: {result['is_gradual']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['cosine_to_final'] * 20)
    print(f"  Layer {p['layer']}: norm={p['norm']:.4f}, "
          f"cos_final={p['cosine_to_final']:.4f}, "
          f"cos_prev={p['cosine_to_previous']:.4f} {bar}")

## Effective Dimension

How many dimensions are actually used by the representations?

In [ ]:
for layer in range(4):
    result = representation_effective_dimension(model, tokens, layer=layer)
    flag = ' [LOW-DIM]' if result['is_low_dimensional'] else ''
    print(f"  Layer {layer}: PR={result['participation_ratio']:.2f}, "
          f"dim90={result['dim_for_90_pct']}, "
          f"top_sv={result['top_singular_value']:.4f}{flag}")

## Geometry Summary

Cross-layer overview of representation geometry.

In [ ]:
result = representation_geometry_summary(model, tokens)
print(f"Geometry trend: {result['geometry_trend']}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: norm={p['mean_norm']:.4f}, "
          f"cos={p['mean_pairwise_cosine']:.4f}, "
          f"PR={p['participation_ratio']:.2f}")